# Guia: Objetos en Python

Esta guia explica los conceptos de programacion orientada a objetos (POO) en Python que necesitaras para el Taller 3 de Logica. No asume conocimientos previos de POO — solo que ya sabes usar listas, diccionarios, funciones y condicionales.

## 1. Clases y Objetos Basicos

Una **clase** es como un molde que define que datos tiene algo y que operaciones puede hacer. Un **objeto** es una instancia concreta de ese molde.

En este ejemplo, la clase `Punto` tiene dos atributos (`x` e `y`) y un metodo (`distancia_al_origen`). Cada vez que creas un `Punto`, estas creando un objeto con esos atributos y metodos.

La clase funciona como un molde: define que es un Punto, pero no es un punto en si mismo. Nos permite saber que siempre que veamos un objeto de tipo Punto, tendra esos atributos y metodos.

- `__init__` es el **constructor**: se ejecuta cuando creas un objeto.
- `self` es una referencia al objeto que se esta creando.
- Un **metodo** es una funcion que pertenece a la clase. Siempre recibe `self` como primer parametro.

**Clave:** Cada objeto tiene sus propios datos (atributos). `p1.x` es 3, pero `p2.x` es 0. Son independientes.

In [ ]:
class Punto:
    """Representa un punto en el plano 2D."""

    def __init__(self, x, y):
        self.x = x
        self.y = y

    def distancia_al_origen(self):
        return (self.x**2 + self.y**2) ** 0.5


# Crear objetos (instancias de la clase Punto):
p1 = Punto(3, 4)
p2 = Punto(0, 0)

print(f"p1.x = {p1.x}, p1.y = {p1.y}")
print(f"p1.distancia_al_origen() = {p1.distancia_al_origen()}")
print(f"p2.distancia_al_origen() = {p2.distancia_al_origen()}")

## 2. Comparar y Representar Objetos (`__eq__`, `__repr__`)

Por defecto, dos objetos son iguales solo si son **el mismo objeto** en memoria. Pero podemos definir `__eq__` para comparar por contenido.

- `__eq__` define cuando dos objetos son "iguales".
- `__repr__` define como se muestra el objeto cuando lo imprimes.
- `isinstance` verifica si un objeto es de un tipo determinado.

In [ ]:
class PuntoMejorado:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __eq__(self, other):
        """Define cuando dos puntos son 'iguales'."""
        if not isinstance(other, PuntoMejorado):
            return False
        return self.x == other.x and self.y == other.y

    def __repr__(self):
        """Define como se muestra el objeto cuando lo imprimes."""
        return f"Punto({self.x}, {self.y})"


a = PuntoMejorado(3, 4)
b = PuntoMejorado(3, 4)
c = PuntoMejorado(1, 2)

print(f"a = {a}")
print(f"b = {b}")
print(f"a == b: {a == b}")  # True (mismo contenido)
print(f"a == c: {a == c}")  # False (distinto contenido)

## 3. Herencia: Clases que Extienden Otras Clases

La herencia permite crear clases que comparten comportamiento. En el taller, **todas** las formulas logicas heredan de una clase base `Formula`.

- `Figura` es la **clase base** (padre).
- `Cuadrado` y `Circulo` son **clases hijas** que heredan de `Figura`.
- Cada clase hija **sobreescribe** el metodo `area()` con su propia implementacion.

**Clave:** Tanto `Cuadrado` como `Circulo` son "Figuras". Puedes tratarlos de la misma manera (llamar `.area()`) aunque hagan cosas distintas.

In [ ]:
class Figura:
    """Clase base para todas las figuras."""

    def area(self):
        raise NotImplementedError("Las subclases deben implementar area()")


class Cuadrado(Figura):
    """Un cuadrado hereda de Figura."""

    def __init__(self, lado):
        self.lado = lado

    def area(self):
        return self.lado**2

    def __repr__(self):
        return f"Cuadrado(lado={self.lado})"


class Circulo(Figura):
    """Un circulo hereda de Figura."""

    def __init__(self, radio):
        self.radio = radio

    def area(self):
        import math
        return math.pi * self.radio**2

    def __repr__(self):
        return f"Circulo(radio={self.radio})"


figuras = [Cuadrado(5), Circulo(3), Cuadrado(2)]
for f in figuras:
    print(f"{f} -> area = {f.area():.2f}")

## 4. `isinstance()` -- Identificar Tipos

`isinstance()` es **fundamental** en el taller. Lo usaras constantemente para determinar que tipo de formula estas procesando.

- Puedes preguntar por la clase especifica: `isinstance(f, Cuadrado)`
- Tambien puedes preguntar por la clase padre: `isinstance(f, Figura)` (siempre `True` para cualquier figura)

In [ ]:
for f in figuras:
    if isinstance(f, Cuadrado):
        print(f"{f} es un Cuadrado, lado = {f.lado}")
    elif isinstance(f, Circulo):
        print(f"{f} es un Circulo, radio = {f.radio}")

    # Tambien puedes preguntar por la clase padre:
    print(f"  Es Figura? {isinstance(f, Figura)}")

## 5. Arboles con Objetos

En el taller, las formulas logicas forman un **arbol**. Por ejemplo: `And(Not(Atom('p')), Atom('q'))`

```
       And
      /   \
    Not    Atom('q')
     |
   Atom('p')
```

Esto se implementa con clases que contienen otros objetos del mismo tipo. Las clases `Expresion`, `Numero`, `Suma` y `Multiplicacion` forman un arbol de expresiones aritmeticas.

La funcion `evaluar` recorre el arbol **recursivamente** usando `isinstance()` para decidir que hacer en cada nodo. Este patron es exactamente lo que haras en el taller con formulas logicas.

In [ ]:
class Expresion:
    """Clase base para expresiones aritmeticas."""
    pass


class Numero(Expresion):
    def __init__(self, valor):
        self.valor = valor

    def __repr__(self):
        return str(self.valor)


class Suma(Expresion):
    def __init__(self, izquierda, derecha):
        self.izquierda = izquierda
        self.derecha = derecha

    def __repr__(self):
        return f"({self.izquierda} + {self.derecha})"


class Multiplicacion(Expresion):
    def __init__(self, izquierda, derecha):
        self.izquierda = izquierda
        self.derecha = derecha

    def __repr__(self):
        return f"({self.izquierda} * {self.derecha})"


# Construir: (3 + 4) * 2
expr = Multiplicacion(
    Suma(Numero(3), Numero(4)),
    Numero(2),
)
print(f"Expresion: {expr}")


# Funcion RECURSIVA que evalua la expresion recorriendo el arbol:
def evaluar(expr):
    if isinstance(expr, Numero):
        return expr.valor
    elif isinstance(expr, Suma):
        return evaluar(expr.izquierda) + evaluar(expr.derecha)
    elif isinstance(expr, Multiplicacion):
        return evaluar(expr.izquierda) * evaluar(expr.derecha)


print(f"Resultado: {evaluar(expr)}")

## 6. Las Clases del Taller (Simplificadas)

Ahora veamos como funcionan las clases reales del taller. No necesitas importarlas -- aqui las recreamos simplificadas.

Las clases son:
- `Formula`: clase base para formulas logicas.
- `Atom`: proposicion atomica (ej: `Atom('llueve')`).
- `Not`: negacion (ej: `Not(Atom('llueve'))`).
- `And`: conjuncion (ej: `And(Atom('p'), Atom('q'))`).
- `Or`: disyuncion (ej: `Or(Atom('p'), Atom('q'))`).
- `Implies`: implicacion (ej: `Implies(Atom('p'), Atom('q'))`).
- `Iff`: bicondicional (ej: `Iff(Atom('p'), Atom('q'))`).

Ejemplo: "Si llueve y no hay sombrilla, entonces me mojo" se traduce a:

```
Implies(And(Atom('llueve'), Not(Atom('sombrilla'))), Atom('me_mojo'))
```

Estructura del arbol:
```
  Implies
  |-- And
  |   |-- Atom('llueve')
  |   +-- Not
  |       +-- Atom('sombrilla')
  +-- Atom('me_mojo')
```

In [ ]:
class Formula:
    """Clase base para formulas logicas."""
    pass


class Atom(Formula):
    """Proposicion atomica. Ejemplo: Atom('llueve')"""

    def __init__(self, name):
        self.name = name

    def __repr__(self):
        return f"Atom('{self.name}')"

    def __eq__(self, other):
        return isinstance(other, Atom) and self.name == other.name


class Not(Formula):
    """Negacion. Ejemplo: Not(Atom('llueve'))"""

    def __init__(self, operand):
        self.operand = operand

    def __repr__(self):
        return f"Not({self.operand!r})"


class And(Formula):
    """Conjuncion. Ejemplo: And(Atom('p'), Atom('q'))"""

    def __init__(self, *conjuncts):
        self.conjuncts = tuple(conjuncts)

    def __repr__(self):
        args = ", ".join(repr(c) for c in self.conjuncts)
        return f"And({args})"


class Or(Formula):
    """Disyuncion. Ejemplo: Or(Atom('p'), Atom('q'))"""

    def __init__(self, *disjuncts):
        self.disjuncts = tuple(disjuncts)

    def __repr__(self):
        args = ", ".join(repr(d) for d in self.disjuncts)
        return f"Or({args})"


class Implies(Formula):
    """Implicacion. Ejemplo: Implies(Atom('p'), Atom('q'))"""

    def __init__(self, antecedent, consequent):
        self.antecedent = antecedent
        self.consequent = consequent

    def __repr__(self):
        return f"Implies({self.antecedent!r}, {self.consequent!r})"


class Iff(Formula):
    """Bicondicional. Ejemplo: Iff(Atom('p'), Atom('q'))"""

    def __init__(self, left, right):
        self.left = left
        self.right = right

    def __repr__(self):
        return f"Iff({self.left!r}, {self.right!r})"


# Construir: "Si llueve y no hay sombrilla, entonces me mojo"
formula = Implies(
    And(Atom("llueve"), Not(Atom("sombrilla"))),
    Atom("me_mojo"),
)
print(f"Formula: {formula}")

## 7. Recorrer Formulas Recursivamente

Este es el patron que usaras en tus implementaciones. La funcion `contar_atomos` cuenta cuantas proposiciones atomicas hay en una formula, recorriendo el AST recursivamente.

El patron general es:
1. Preguntar que tipo de nodo es (`isinstance`).
2. **Caso base**: si es un `Atom`, devolver el resultado directamente.
3. **Caso recursivo**: aplicar la funcion a los hijos del nodo.
   - `Not` tiene un solo hijo: `formula.operand`
   - `And` tiene multiples hijos: `formula.conjuncts` (una tupla)
   - `Or` tiene multiples hijos: `formula.disjuncts` (una tupla)
   - `Implies` tiene dos hijos: `formula.antecedent` y `formula.consequent`
   - `Iff` tiene dos hijos: `formula.left` y `formula.right`

In [ ]:
def contar_atomos(formula):
    """
    Cuenta cuantas proposiciones atomicas hay en una formula.
    Ejemplo de como recorrer el AST recursivamente.
    """
    if isinstance(formula, Atom):
        return 1

    elif isinstance(formula, Not):
        return contar_atomos(formula.operand)

    elif isinstance(formula, And):
        total = 0
        for conjunct in formula.conjuncts:
            total += contar_atomos(conjunct)
        return total

    elif isinstance(formula, Or):
        total = 0
        for disjunct in formula.disjuncts:
            total += contar_atomos(disjunct)
        return total

    elif isinstance(formula, Implies):
        return contar_atomos(formula.antecedent) + contar_atomos(formula.consequent)

    elif isinstance(formula, Iff):
        return contar_atomos(formula.left) + contar_atomos(formula.right)

    return 0


print(f"Formula: {formula}")
print(f"Numero de atomos: {contar_atomos(formula)}")

## 8. Transformar Formulas -- Patron del Taller

En el taller debes transformar formulas. El patron es:
1. Preguntar que tipo de nodo es (`isinstance`).
2. Si es el tipo que quieres transformar, aplicar la regla.
3. Si no, aplicar recursivamente a los hijos y reconstruir el nodo.

La funcion `eliminar_doble_negacion` transforma `Not(Not(x))` en `x`. Este es uno de los ejercicios del taller.

In [ ]:
def eliminar_doble_negacion(formula):
    """
    Transforma Not(Not(x)) -> x

    Este es uno de los ejercicios del taller. Aqui te mostramos
    el patron para que entiendas la estructura.
    """
    # Caso base: un atomo no tiene nada que transformar
    if isinstance(formula, Atom):
        return formula

    # Aqui esta la transformacion!
    if isinstance(formula, Not):
        # Verificar si es doble negacion: Not(Not(algo))
        if isinstance(formula.operand, Not):
            # Not(Not(x)) -> x, pero aplicar recursivamente a x
            return eliminar_doble_negacion(formula.operand.operand)
        # Si no es doble negacion, solo aplicar recursivamente al hijo
        return Not(eliminar_doble_negacion(formula.operand))

    # Para And, aplicar a cada hijo y reconstruir
    if isinstance(formula, And):
        nuevos = tuple(eliminar_doble_negacion(c) for c in formula.conjuncts)
        return And(*nuevos)

    # Para Or, igual
    if isinstance(formula, Or):
        nuevos = tuple(eliminar_doble_negacion(d) for d in formula.disjuncts)
        return Or(*nuevos)

    # Para Implies y Iff, aplicar a cada parte
    if isinstance(formula, Implies):
        return Implies(
            eliminar_doble_negacion(formula.antecedent),
            eliminar_doble_negacion(formula.consequent),
        )

    if isinstance(formula, Iff):
        return Iff(
            eliminar_doble_negacion(formula.left),
            eliminar_doble_negacion(formula.right),
        )

    return formula


# Ejemplo 1: Not(Not(Atom('p'))) -> Atom('p')
f1 = Not(Not(Atom("p")))
print(f"Original:     {f1}")
print(f"Transformada: {eliminar_doble_negacion(f1)}")

# Ejemplo 2: And(Not(Not(Atom('p'))), Atom('q')) -> And(Atom('p'), Atom('q'))
f2 = And(Not(Not(Atom("p"))), Atom("q"))
print(f"Original:     {f2}")
print(f"Transformada: {eliminar_doble_negacion(f2)}")

# Ejemplo 3: Not(Not(Not(Atom('p')))) -> Not(Atom('p'))
f3 = Not(Not(Not(Atom("p"))))
print(f"Original:     {f3}")
print(f"Transformada: {eliminar_doble_negacion(f3)}")

## 9. Dataclasses Frozen -- Para Predicados

En la parte de logica de predicados, el codigo usa `@dataclass(frozen=True)`. Esto es como una clase normal pero mas concisa y el objeto **no se puede modificar** despues de crearlo (inmutabilidad).

- `frozen=True` significa que NO puedes cambiar los atributos despues de crear el objeto.
- `@dataclass` genera automaticamente `__init__`, `__eq__`, `__repr__` y `__hash__`.
- Las **variables** empiezan con mayuscula, las **constantes** con minuscula.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Termino:
    """
    Un termino: variable o constante.

    frozen=True significa que NO puedes cambiar los atributos despues de crear el objeto.
    @dataclass genera automaticamente __init__, __eq__, __repr__ y __hash__.
    """

    nombre: str

    @property
    def es_variable(self):
        """Las variables empiezan con mayuscula."""
        return self.nombre[0].isupper()


@dataclass(frozen=True)
class Predicado:
    """
    Un predicado con argumentos.
    Ejemplo: tiene_motivo(mayordomo)
    """

    nombre: str
    args: tuple  # tupla de Terminos

    def __repr__(self):
        args_str = ", ".join(str(a.nombre) for a in self.args)
        return f"{self.nombre}({args_str})"


# Los dataclass generan el __init__ automaticamente:
t1 = Termino("mayordomo")  # constante (empieza con minuscula)
t2 = Termino("X")          # variable (empieza con mayuscula)

print(f"t1 = {t1}, es_variable = {t1.es_variable}")
print(f"t2 = {t2}, es_variable = {t2.es_variable}")

# Comparacion automatica:
t3 = Termino("mayordomo")
print(f"t1 == t3: {t1 == t3}")  # True (mismo contenido)

# Frozen = no puedes modificar:
try:
    t1.nombre = "otro"
except AttributeError as e:
    print(f"Error al modificar: {e}")

# Crear predicados como en el taller:
p1 = Predicado("tiene_motivo", (Termino("mayordomo"),))
p2 = Predicado("persona", (Termino("X"),))

print(f"p1 = {p1}")
print(f"p2 = {p2}")

## 10. Clausulas de Horn -- Modelar Reglas

En el taller, modelaras reglas como:

> "Una persona es sospechosa si tiene motivo y oportunidad"

Esto se traduce a:

```
sospechoso(X) :- persona(X), tiene_motivo(X), tiene_oportunidad(X)
```

- `X` es una **variable** (mayuscula) que se puede sustituir por cualquier valor.
- `persona`, `tiene_motivo` son nombres de predicados.
- La regla dice: para cualquier X, si X es persona Y tiene motivo Y tiene oportunidad, ENTONCES X es sospechoso.

En el taller, el codigo del motor (`forward_chaining.py` y `backward_chaining.py`) ya implementa la logica de unificacion y encadenamiento. Tu trabajo es **modelar las reglas correctamente** usando estos objetos.

In [ ]:
@dataclass(frozen=True)
class Regla:
    """
    Una regla (clausula de Horn): cabeza :- cuerpo.

    La cabeza se cumple SI se cumplen todos los predicados del cuerpo.
    """

    cabeza: Predicado
    cuerpo: tuple  # tupla de Predicados

    def __repr__(self):
        cuerpo_str = ", ".join(str(b) for b in self.cuerpo)
        return f"{self.cabeza} :- {cuerpo_str}."


# Ejemplo: "Una persona es sospechosa si tiene motivo y oportunidad"
regla = Regla(
    cabeza=Predicado("sospechoso", (Termino("X"),)),
    cuerpo=(
        Predicado("persona", (Termino("X"),)),
        Predicado("tiene_motivo", (Termino("X"),)),
        Predicado("tiene_oportunidad", (Termino("X"),)),
    ),
)
print(f"Regla: {regla}")

## Resumen: Conceptos Clave para el Taller

1. **CLASE**: Un molde que define atributos y metodos.
   Ejemplo: `class Atom(Formula): ...`

2. **OBJETO**: Una instancia concreta de una clase.
   Ejemplo: `Atom('llueve')`

3. **ATRIBUTOS**: Los datos que tiene un objeto.
   Ejemplo: `atom.name`, `not_formula.operand`, `and_formula.conjuncts`

4. **isinstance()**: Pregunta "este objeto es de este tipo?"
   Ejemplo: `isinstance(formula, Atom)` -> `True`/`False`

5. **HERENCIA**: `Formula` es la clase padre. `Atom`, `Not`, `And`, `Or`, `Implies`, `Iff` son clases hijas. Todas son `Formula`.

6. **RECURSION SOBRE EL AST**: El patron central del taller.
   Para cada tipo de nodo:
   - `Atom`: caso base
   - `Not`: procesar `formula.operand`
   - `And`: procesar cada elemento de `formula.conjuncts`
   - `Or`: procesar cada elemento de `formula.disjuncts`
   - `Implies`: procesar `formula.antecedent` y `formula.consequent`
   - `Iff`: procesar `formula.left` y `formula.right`

7. **ATRIBUTOS POR TIPO**:

| Tipo | Atributos |
|------|-----------|
| `Atom` | `.name` (str) |
| `Not` | `.operand` (Formula) |
| `And` | `.conjuncts` (tupla de Formulas) |
| `Or` | `.disjuncts` (tupla de Formulas) |
| `Implies` | `.antecedent`, `.consequent` (Formulas) |
| `Iff` | `.left`, `.right` (Formulas) |

8. **DATACLASS + FROZEN**: Objetos inmutables con `__init__` automatico. Usados en logica de predicados: `Term`, `Predicate`, `Fact`, `Rule`.

9. **CLAUSULAS DE HORN**: `Regla(cabeza, cuerpo)`. La cabeza se cumple si TODOS los predicados del cuerpo se cumplen.